In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from torchvision import datasets, transforms

# 1. Point to our classification folders
train_dir = '/content/drive/MyDrive/Brain_Tumor_Project/brisc2025/classification_task/train'
test_dir = '/content/drive/MyDrive/Brain_Tumor_Project/brisc2025/classification_task/test'

# 2. Give the computer instructions on how to prepare the pictures
image_rules = transforms.Compose([
    transforms.Resize((224, 224)), # Make every picture 224x224 pixels
    transforms.ToTensor()          # Turn the picture into math (numbers)
])

# 3. Apply the rules to our folders!
train_data = datasets.ImageFolder(train_dir, transform=image_rules)
test_data = datasets.ImageFolder(test_dir, transform=image_rules)

# 4. Let's see what we got
print("Pictures ready for training:", len(train_data))
print("Pictures hidden away for testing:", len(test_data))
print("Categories the AI will learn:", train_data.classes)

Pictures ready for training: 5000
Pictures hidden away for testing: 1000
Categories the AI will learn: ['glioma', 'meningioma', 'no_tumor', 'pituitary']


In [ ]:
from torch.utils.data import DataLoader

# How many pictures to feed the AI at once
batch_size = 32

# Create our "flashcard" stacks
# We shuffle the training data so the AI doesn't memorize the order
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

# We don't need to shuffle the test data, because we are just grading it
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print("DataLoaders are ready! The AI will look at pictures in groups of", batch_size)

DataLoaders are ready! The AI will look at pictures in groups of 32


In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch

# 1. Download the pre-built "smart student" brain
print("Downloading ResNet50...")
ai_model = models.resnet50(weights="DEFAULT")

# 2. ResNet50 originally learned 1,000 different things (dogs, cars, etc.)
# We need to erase its old answer sheet and give it a new one with only 4 options (our 4 tumor types)
number_of_features = ai_model.fc.in_features
ai_model.fc = nn.Linear(number_of_features, 4)

# 3. Tell the brain to use our super-fast GPU engine
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ai_model = ai_model.to(device)

print("AI Brain is ready!")
print("It is running on:", device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 152MB/s]


AI Brain is ready!
It is running on: cuda


In [ ]:
import torch.optim as optim
import torch.nn as nn

# 1. The Grader (CrossEntropyLoss is perfect for multiple-choice categories)
criterion = nn.CrossEntropyLoss()

# 2. The Tutor (Adam is a very popular, smart optimizer)
# lr=0.001 is the "learning rate" - it means the tutor makes small, careful adjustments
optimizer = optim.Adam(ai_model.parameters(), lr=0.001)

print("Grader and Tutor are ready!")

Grader and Tutor are ready!


In [ ]:
# This copies the folder from the slow Drive to the fast Colab memory
!cp -r /content/drive/MyDrive/Brain_Tumor_Project/brisc2025/classification_task /content/fast_data

print("Copying complete! The pictures are now in fast memory.")

Copying complete! The pictures are now in fast memory.


In [ ]:
# 1. Point to the new FAST memory folders
fast_train_dir = '/content/fast_data/train'

# 2. Rebuild our flashcard stacks using the fast folder
fast_train_data = datasets.ImageFolder(fast_train_dir, transform=image_rules)
fast_train_loader = DataLoader(fast_train_data, batch_size=batch_size, shuffle=True)

print("Map updated! Starting the FAST training...\n")

# 3. The Training Loop (exactly the same as before, but using fast_train_loader)
for epoch in range(epochs):
    print(f"--- Starting Epoch {epoch+1} of {epochs} ---")
    ai_model.train()
    total_mistakes = 0.0

    for i, (images, labels) in enumerate(fast_train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        guesses = ai_model(images)
        mistake = criterion(guesses, labels)
        mistake.backward()
        optimizer.step()

        total_mistakes += mistake.item()

        # Print an update every 50 batches
        if i % 50 == 0:
            print(f"Batch {i}: AI is studying...")

    average_mistake = total_mistakes / len(fast_train_loader)
    print(f"Epoch {epoch+1} finished! Average Mistake Score: {average_mistake:.4f}\n")

print("Training complete! The AI has a much smarter brain now.")

Map updated! Starting the FAST training...

--- Starting Epoch 1 of 3 ---
Batch 0: AI is studying...
Batch 50: AI is studying...
Batch 100: AI is studying...
Batch 150: AI is studying...
Epoch 1 finished! Average Mistake Score: 0.2270

--- Starting Epoch 2 of 3 ---
Batch 0: AI is studying...
Batch 50: AI is studying...
Batch 100: AI is studying...
Batch 150: AI is studying...
Epoch 2 finished! Average Mistake Score: 0.1078

--- Starting Epoch 3 of 3 ---
Batch 0: AI is studying...
Batch 50: AI is studying...
Batch 100: AI is studying...
Batch 150: AI is studying...
Epoch 3 finished! Average Mistake Score: 0.0952

Training complete! The AI has a much smarter brain now.


In [ ]:
import torch

# This is exactly where we want to save the brain in your Google Drive
save_path = '/content/drive/MyDrive/Brain_Tumor_Project/classifier.pth'

# This takes the learned weights (the memory) and saves it to a file
torch.save(ai_model.state_dict(), save_path)

print("AI Brain safely saved to Google Drive!")

AI Brain safely saved to Google Drive!
